In [ ]:
!pip install nltk gensim node2vec karateclub transformers torch tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of karateclub to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
!pip install nltk gensim node2vec tensorflow scikit-learn

  Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
  Using cached node2vec-0.5.0-py3-none-any.whl.metadata (849 bytes)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which

In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
import nltk
import networkx as nx

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate
from gensim.models import Word2Vec
from node2vec import Node2Vec

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
zip_path = "rumor_detection_acl2017.zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done")


Extraction Done


In [ ]:
DATASET_NAME = "twitter15"   # change to twitter16
BASE_PATH = "/content/rumor_detection_acl2017"
DATA_PATH = f"{BASE_PATH}/{DATASET_NAME}"

print("Using dataset:", DATA_PATH)

Using dataset: /content/rumor_detection_acl2017/twitter15


In [ ]:
labels = {}

with open(f"{DATA_PATH}/label.txt") as f:
    for line in f:
        label, tweet_id = line.strip().split(":")
        if label == "true":
            labels[tweet_id] = 1
        elif label == "false":
            labels[tweet_id] = 0

In [ ]:
texts = {}

with open(f"{DATA_PATH}/source_tweets.txt") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        tweet_id, text = parts[0], parts[1]
        if tweet_id in labels:
            texts[tweet_id] = text

In [ ]:
data = []

for tid in texts:
    data.append((tid, texts[tid], labels[tid]))

df = pd.DataFrame(data, columns=["id", "text", "label"])
print("Dataset size:", len(df))


Dataset size: 742


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = text.lower()
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return tokens


df["tokens"] = df["text"].apply(clean_text)

In [ ]:
sentences = df["tokens"].tolist()

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=10,
    min_count=4,
    sg=1,
    epochs=5
)


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["tokens"].apply(lambda x: " ".join(x)))

sequences = tokenizer.texts_to_sequences(df["tokens"].apply(lambda x: " ".join(x)))
X_text = pad_sequences(sequences, maxlen=30)


In [ ]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]


In [ ]:
G = nx.Graph()
tree_path = f"{DATA_PATH}/tree"

for file in os.listdir(tree_path):
    with open(f"{tree_path}/{file}") as f:
        for line in f:
            try:
                parent, child = line.strip().split("->")
                G.add_edge(parent.strip(), child.strip())
            except:
                continue

print("Graph nodes:", len(G.nodes()))

Graph nodes: 599197


In [ ]:
node2vec = Node2Vec(G, dimensions=100, walk_length=10, num_walks=10)
model_n2v = node2vec.fit(window=10, min_count=1)

node_embeddings = {}
for node in G.nodes():
    node_embeddings[node] = model_n2v.wv[node]

Computing transition probabilities:   0%|          | 0/599197 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 10/10 [32:36<00:00, 195.63s/it]


In [13]:
def get_node_embedding(tweet_id):
    return node_embeddings.get(tweet_id, np.zeros(100))

X_graph = np.array([get_node_embedding(i) for i in df["id"]])


In [14]:
y = df["label"].values

In [15]:
X_text_train, X_text_test, X_graph_train, X_graph_test, y_train, y_test = train_test_split(
    X_text, X_graph, y, test_size=0.2, random_state=42
)


In [16]:
text_input = Input(shape=(30,))
embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    trainable=False
)(text_input)

text_branch = LSTM(64)(embedding_layer)

graph_input = Input(shape=(100,))
graph_branch = Dense(32, activation='relu')(graph_input)

combined = Concatenate()([text_branch, graph_branch])
x = Dense(32, activation='relu')(combined)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[text_input, graph_input], outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 100)   │    244,300 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     42,240 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      3,232 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      3,104 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 292,909 (1.12 MB)

 Trainable params: 48,609 (189.88 KB)

 Non-trainable params: 244,300 (954.30 KB)

In [17]:
model.fit(
    [X_text_train, X_graph_train],
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.4991 - loss: 0.6893 - val_accuracy: 0.5167 - val_loss: 0.6816
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5516 - loss: 0.6815 - val_accuracy: 0.6000 - val_loss: 0.6662
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.6135 - loss: 0.6731 - val_accuracy: 0.6500 - val_loss: 0.6478
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.6285 - loss: 0.6548 - val_accuracy: 0.7000 - val_loss: 0.6365
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.6360 - loss: 0.6518 - val_accuracy: 0.6667 - val_loss: 0.6326
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.6266 - loss: 0.6478 - val_accuracy: 0.6833 - val_loss: 0.6317
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6229 - loss: 0.6536 - val_accuracy: 0.6500 - val_loss: 0.6352
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6454 - loss: 0.6423 - val_accuracy: 0.6667 - v

In [18]:
loss, acc = model.evaluate([X_text_test, X_graph_test], y_test)
print("Final Accuracy:", acc)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6040 - loss: 0.6858
Final Accuracy: 0.6040268540382385
